# 🤖 NLP Assignment – Task 3: Chatbot using Hugging Face Transformers

---

## Objective
Build a console-based conversational chatbot using the **DialoGPT** pre-trained transformer model from Hugging Face. The chatbot dynamically generates responses and maintains multi-turn conversation context.

## Pipeline Flow
```
User Input → Tokenization → Model Processing → Response Generation → Display Output → Loop Until Exit
```

## Step 1: Install Required Libraries

We install the `transformers` and `torch` libraries needed to load and run the pre-trained model.

In [ ]:
# Install Hugging Face Transformers and PyTorch (if not already installed)
! pip install transformers torch --quiet

## Step 2: Import Libraries

We import the necessary modules:
- `AutoModelForCausalLM` – loads the pre-trained causal language model (DialoGPT)
- `AutoTokenizer` – converts text to token IDs the model understands
- `torch` – PyTorch backend for tensor operations

In [ ]:
# Import required libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Libraries imported successfully.")

 Libraries imported successfully.


## Step 3: Load Pre-Trained Model and Tokenizer

We use **DialoGPT-medium** — a dialogue-focused GPT-2 variant fine-tuned on conversational data from Reddit. It is well-suited for multi-turn chatbot tasks.

- The **tokenizer** encodes text into token IDs.
- The **model** generates the next token IDs based on the conversation history.

In [ ]:
# Define the model name from Hugging Face Model Hub
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"Loading tokenizer and model: {MODEL_NAME} ...")

# Load the tokenizer — converts raw text to token IDs
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the pre-trained causal language model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set model to evaluation mode (disables dropout, etc.)
model.eval()

print("Model and tokenizer loaded successfully!")

⏳ Loading tokenizer and model: microsoft/DialoGPT-medium ...


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--microsoft--DialoGPT-medium. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

## Step 4: Define the Response Generation Function

This function:
1. Encodes the user's input and appends it to the conversation history
2. Passes the full history to the model for context-aware generation
3. Decodes the model's output tokens back into human-readable text
4. Returns both the response text and the updated conversation history

In [ ]:
def generate_response(user_input, conversation_history, max_history_length=5):
    """
    Generate a chatbot response using DialoGPT.

    Args:
        user_input (str): The latest message from the user.
        conversation_history (list): List of past token ID tensors for context.
        max_history_length (int): Maximum number of past turns to retain.

    Returns:
        response_text (str): The chatbot's reply.
        conversation_history (list): Updated conversation history.
    """

    # Encode the user input and append EOS token (marks end of user turn)
    user_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"  # Return as PyTorch tensor
    )

    # Append new user input to conversation history
    conversation_history.append(user_input_ids)

    # Trim history to prevent exceeding the model's max token limit
    if len(conversation_history) > max_history_length * 2:
        conversation_history = conversation_history[-(max_history_length * 2):]

    # Concatenate all turns in history into a single input tensor
    bot_input_ids = torch.cat(conversation_history, dim=-1)

    # Generate response using the model
    # - max_new_tokens: limits the length of the generated reply
    # - pad_token_id: prevents warnings about missing padding token
    # - do_sample: enables sampling for diverse responses
    # - top_k / top_p: nucleus + top-k sampling for quality control
    # - temperature: controls randomness (lower = more deterministic)
    with torch.no_grad():  # Disable gradient computation for inference
        output_ids = model.generate(
            bot_input_ids,
            max_new_tokens=150,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.92,
            temperature=0.75,
            repetition_penalty=1.3
        )

    # Decode only the newly generated tokens (skip the input tokens)
    new_tokens = output_ids[:, bot_input_ids.shape[-1]:]
    response_text = tokenizer.decode(new_tokens[0], skip_special_tokens=True)

    # Append the model's response to history for future context
    conversation_history.append(output_ids[:, bot_input_ids.shape[-1]:])

    return response_text, conversation_history


print("Response generation function defined.")

## Step 5: Run the Interactive Chatbot

The chatbot loop:
- Greets the user on start
- Continuously accepts user input
- Generates and displays responses
- Exits gracefully when the user types `exit` or `quit`

> **Note:** In Jupyter Notebook, the `input()` function creates an interactive prompt in the output cell. Type your message and press Enter to chat.

In [ ]:
def run_chatbot():
    """
    Main function to run the interactive console-based chatbot.
    Maintains conversation history across turns for contextual replies.
    Exits when the user types 'exit' or 'quit'.
    """

    print("=" * 55)
    print("       🤖 DialoGPT Chatbot — Powered by Transformers")
    print("=" * 55)
    print("  Type your message and press Enter to chat.")
    print("  Type 'exit' or 'quit' to end the conversation.")
    print("-" * 55)

    # Greeting message from the chatbot
    print("\nChatbot: Hello! I am your AI assistant. How can I help you today?\n")

    # Initialize empty conversation history
    conversation_history = []

    # Main conversation loop
    while True:
        # Get user input
        user_input = input("You: ").strip()

        # Skip empty inputs
        if not user_input:
            print("Chatbot: Please type something so I can help you!\n")
            continue

        # Check for exit keywords (case-insensitive)
        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot: It was great talking with you! Goodbye! 👋")
            print("-" * 55)
            break

        # Generate and display the chatbot's response
        response, conversation_history = generate_response(
            user_input, conversation_history
        )

        # Handle edge case where model returns an empty response
        if not response.strip():
            response = "I'm not sure how to respond to that. Could you rephrase?"

        print(f"\nChatbot: {response}\n")


# Run the chatbot
run_chatbot()

## Step 6: Sample Chatbot Interaction Output

Below is a simulated demo of the chatbot interaction (for reference, as `input()` does not execute in static view):

In [ ]:
# -------------------------------------------------------
# DEMO: Simulate a conversation without live input()
# This cell showcases chatbot responses for submission.
# -------------------------------------------------------

def demo_chatbot(user_messages):
    """
    Run a non-interactive demo with a predefined list of user messages.
    Useful for showing chatbot output in a static Jupyter Notebook view.

    Args:
        user_messages (list): List of user input strings to simulate.
    """
    print("=" * 55)
    print("       🤖 DialoGPT Chatbot Demo — Sample Interaction")
    print("=" * 55)
    print("\nChatbot: Hello! I am your AI assistant. How can I help you today?\n")

    conversation_history = []

    for user_input in user_messages:
        print(f"You: {user_input}")

        # Check exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot: It was great talking with you! Goodbye! 👋")
            print("-" * 55)
            break

        # Generate response
        response, conversation_history = generate_response(
            user_input, conversation_history
        )

        if not response.strip():
            response = "I'm not sure how to respond to that. Could you rephrase?"

        print(f"\nChatbot: {response}\n")


# Sample conversation messages
demo_messages = [
    "Hello",
    "What is Artificial Intelligence?",
    "Who created Python?",
    "Tell me about machine learning.",
    "Thank you!",
    "exit"
]

demo_chatbot(demo_messages)

In [ ]:
print("Example Conversation:\n")

print("You: Hello!")
print("Bot: Hi there! How can I help you?\n")

print("You: What is AI?")
print("Bot: Artificial Intelligence is...")